In [ ]:
import pandas as pd

path = 'all_beauty/profiles/train/item_profile.parquet'
df_train = pd.read_parquet(path)

In [ ]:
path = 'all_beauty/splits/test.parquet'
df_test = pd.read_parquet(path)

In [ ]:
df_test


In [ ]:
import re

# O parser original usava `:\s*(.+)$`; como `\s` casa com "\n", um campo vazio
# ("Dislikes:  \nContext: ...") capturava a linha seguinte. Aqui o espaço
# após os dois-pontos fica restrito à mesma linha ([ \t]) e campo vazio vira "none".
USER_KEYS = ["Likes", "Dislikes", "Context", "Summary"]
ITEM_KEYS = ["Strengths", "Weaknesses", "Suits", "Summary"]

def match_field(text, k):
    return re.search(rf"^\s*\**{k}\**[ \t]*:[ \t]*(.*)$", text, flags=re.I | re.M)

def parse_fields(text, keys):
    out = {}
    for k in keys:
        m = match_field(text, k)
        value = m.group(1).strip() if m else ""
        out[k.lower()] = value if value else "none"
    out["parse_ok"] = all(match_field(text, k) for k in keys)
    return pd.Series(out)

id_col, keys = ("userId", USER_KEYS) if "userId" in df.columns else ("itemId", ITEM_KEYS)
cols = [k.lower() for k in keys]
fixed = df["raw_output"].apply(parse_fields, keys=keys)
changed = (fixed[cols] != df[cols]).any(axis=1)
print(f"linhas corrigidas: {changed.sum()} de {len(df)}")

df[cols + ["parse_ok"]] = fixed[cols + ["parse_ok"]]
df[df[id_col] == ("23" if id_col == "userId" else "1160")].T

In [ ]:
df.to_parquet(path, index=False)

In [ ]:
df

In [ ]:
df = pd.read_parquet('all_beauty/splits/test.parquet')

In [ ]:
df